# LangGraph Agentic RAG

This notebook runs the real document QA pipeline through the Self-RAG retrieval gate and passage relevance grader. It loads indexed chunks, retrieves and reranks candidates, filters irrelevant passages, and calls the configured LLM.

The graph prepares bounded conversation context, decides whether retrieval is needed, rewrites the query, and then runs the existing RAG query path:

```text
START -> context_manager -> retrieval_gate[Ret] -> query_rewriter -> retrieve -> grade_relevance[Rel] -> generate/abstain -> persist_turn -> END
```

This notebook covers the first two Self-RAG reflections: [Ret] and [Rel]. It does not implement [Sup], [Use], or retrieval retry.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from within the project directory tree.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

from src.agent.context import ContextManager
from src.agent.graph import build_agent_graph, invoke_agent_graph
from src.agent.relevance import LLMRelevanceGrader
from src.agent.rewrite import LLMQueryRewriter
from src.agent.routes import LLMRetrievalGate
from src.core.config import Config
from src.core.logger import setup_logging
from src.pipeline.query_runtime import build_query_pipeline

## Build the real RAG pipeline

This cell uses the same online query-runtime construction path as `main.py`. It loads existing chunks and document embeddings from Chroma, then rebuilds only the in-memory BM25 index. It does not parse documents or re-embed document chunks.

In [ ]:
config = Config()
setup_logging(config)
pipeline = build_query_pipeline(config)

checkpointer = InMemorySaver()
graph = build_agent_graph(
    pipeline,
    retrieval_gate=LLMRetrievalGate(pipeline.llm),
    context_manager=ContextManager(config),
    query_rewriter=LLMQueryRewriter(pipeline.llm),
    relevance_grader=LLMRelevanceGrader(pipeline.llm),
    checkpointer=checkpointer,
)

## Visualize the LangGraph workflow

`draw_mermaid()` exposes the compiled graph structure. LangSmith records executions of this graph when `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` are configured in `.env`.

In [ ]:
mermaid_graph = graph.get_graph().draw_mermaid()
display(Markdown(f'```mermaid\n{mermaid_graph}\n```'))

## Run a two-turn conversation

Both calls use the same `thread_id`. The second request uses a pronoun, so the Query Rewriter must resolve the referenced model from the first turn before document retrieval. Each turn performs query embedding, hybrid retrieval, reranking, and LLM generation against the existing index. The current `InMemorySaver` is not durable across Python processes.

In [ ]:
thread_id = 'notebook-demo'
first_question = 'What classification accuracy did the ResNet26-V2 model achieve?'
second_question = 'What optimizer did it use?'

first_response = invoke_agent_graph(
    graph,
    first_question,
    thread_id=thread_id,
)

second_response = invoke_agent_graph(
    graph,
    second_question,
    thread_id=thread_id,
)

display(Markdown(f'**Turn 1 answer**: {first_response.answer}'))
display(Markdown(f'**Turn 2 answer**: {second_response.answer}'))
for source in second_response.sources:
    print(f'- {source.chunk_id} | {source.source} | page {source.page}')

## Inspect checkpointed state

The checkpoint stores the state after graph execution, including the retrieval action, original and rewritten query, retrieved chunks, and bounded conversation history. Confirm that `original_query` is the second request and `rewritten_query` resolves `it` to `ResNet26-V2`.

In [ ]:
graph_config = {'configurable': {'thread_id': thread_id}}
snapshot = graph.get_state(graph_config)
snapshot.values

## Check LangSmith configuration safely

This cell checks whether tracing is configured without printing the API key. When enabled, open the `doc-qa-agent` project in the LangSmith website to inspect the retrieval gate decision and selected graph branch.

In [ ]:
import os


langsmith_status = {
    'tracing_enabled': os.getenv('LANGSMITH_TRACING', '').lower() == 'true',
    'api_key_configured': bool(os.getenv('LANGSMITH_API_KEY')),
    'project': os.getenv('LANGSMITH_PROJECT', 'default'),
}
langsmith_status

## Runtime requirements

Build the index first with `python -m scripts.build_index` whenever source documents, parsing, chunking, filtering, or the embedding model changes. The online notebook requires `SCADS_API_KEY` for reranking and generation. It needs `MINERU_API_TOKEN` only when building the index. Configure `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, and `LANGSMITH_PROJECT=doc-qa-agent` to view traces in LangSmith. Do not run indexing against sensitive documents unless uploading their content to MinerU and ScaDS.AI is permitted.